### Childhood Autistic Spectrum Disorder Screening using Machine Learning

The early diagnosis of neurodevelopment disorders can improve treatment and significantly decrease the associated
healthcare costs. In this project, we will use supervised learning to diagnose Autistic Spectrum Disorder
(ASD) based on behavioural features and individual characteristics. More specifically, we will build and deploy a neural network using the Keras API.

This project will use a dataset provided by the UCI Machine Learning Repository that contains screening data for 292 patients. The dataset can be found at the following URL:
https://archive.ics.uci.edu/ml/datasets/Autistic+Spectrum+Disorder+Screening+Data+for+Children++

Let's dive right in! First, we will import a few of libraries we will use in this project.

In [1]:
import sys
import pandas as pd
import sklearn
import keras
from sklearn import model_selection
from sklearn.metrics import classification_report, accuracy_score ##
from keras.models import Sequential
from keras.layers import Dense
from keras.optimizers import Adam
from scipy.io import arff

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("Sklearn:", sklearn.__version__)
print("Keras:", keras.__version__)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Pandas: 2.2.2
Sklearn: 1.6.1
Keras: 3.13.2


### 1. Importing the Dataset

We will obtain the data from the UCI Machine Learning Repository; however, since the data isn't contained in a csv or txt file, we will have to download the compressed zip file and then extract the data manually. Once that is accomplished, we will read the information in from a text file using Pandas.

In [2]:
# 2. Fetch Dataset from UCI Repository

!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo

# fetch dataset
autism_data = fetch_ucirepo(id=419)

# extract features and targets
X_raw = autism_data.data.features
y_raw = autism_data.data.targets

# metadata and variable info (optional)
print(autism_data.metadata)
print(autism_data.variables)

{'uci_id': 419, 'name': 'Autistic Spectrum Disorder Screening Data for Children  ', 'repository_url': 'https://archive.ics.uci.edu/dataset/419/autistic+spectrum+disorder+screening+data+for+children', 'data_url': 'https://archive.ics.uci.edu/static/public/419/data.csv', 'abstract': 'Children screening data for autism suitable for classification and predictive tasks ', 'area': 'Health and Medicine', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 292, 'num_features': 20, 'feature_types': ['Integer'], 'demographics': ['\x00', 'Age', 'Gender', 'Ethnicity', 'Nationality'], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2017, 'last_updated': 'Wed Apr 03 2024', 'dataset_doi': '10.24432/C5659W', 'creators': ['Fadi Thabtah'], 'intro_paper': None, 'additional_info': {'summary': "see attached file for variables' description ", 'purpose': None, 'funded_by': None, 'instances_repr

In [3]:
# print the shape of the DataFrame, so we can see how many examples we have
print("Shape of features:", X_raw.shape)
print("Shape of targets:", y_raw.shape)

Shape of features: (292, 20)
Shape of targets: (292, 1)


### 2. Data Preprocessing

This dataset is going to require multiple preprocessing steps. First, we have columns in our DataFrame (attributes) that we don't want to use when training our neural network. We will drop these columns first. Secondly, much of our data is reported using strings; as a result, we will convert our data to categorical labels. During our preprocessing, we will also split the dataset into X and Y datasets, where X has all of the attributes we want to use for prediction and Y has the class labels.

In [4]:
# 3. Data Preprocessing
# Convert categorical features to one-hot encoding
X = pd.get_dummies(X_raw)
# Convert target to one-hot encoding
Y = pd.get_dummies(y_raw)

print("Example patient data (X):\n", X.iloc[0])
print("Class data (Y):\n", Y.head(10))

Example patient data (X):
 A1_Score                                   1
A2_Score                                   1
A3_Score                                   0
A4_Score                                   0
A5_Score                                   1
                                       ...  
relation_'Health care professional'    False
relation_Parent                         True
relation_Relative                      False
relation_Self                          False
relation_self                          False
Name: 0, Length: 88, dtype: object
Class data (Y):
    class_NO  class_YES
0      True      False
1      True      False
2      True      False
3      True      False
4     False       True
5      True      False
6     False       True
7     False       True
8     False       True
9      True      False


### 3. Split the Dataset into Training and Testing Datasets

Before we can begin training our neural network, we need to split the dataset into training and testing datasets. This will allow us to test our network after we are done training to determine how well it will generalize to new data. This step is incredibly easy when using the train_test_split() function provided by scikit-learn!

In [5]:
# 4. Split into Training and Testing Sets
X_train, X_test, Y_train, Y_test = model_selection.train_test_split(X, Y, test_size = 0.2, random_state=42)

In [6]:
print(X_train.shape, X_test.shape, Y_train.shape, Y_test.shape)


(233, 88) (59, 88) (233, 2) (59, 2)


### 4. Building the Network - Keras

In this project, we are going to use Keras to build and train our network. This model will be relatively simple and will only use dense (also known as fully connected) layers. This is the most common neural network layer. The network will have one hidden layer, use an Adam optimizer, and a categorical crossentropy loss. We won't worry about optimizing parameters such as learning rate, number of neurons in each layer, or activation functions in this project; however, if you have the time, manually adjusting these parameters and observing the results is a great way to learn about their function!

In [7]:
# 5. Build Keras Neural Network

# define a function to build the keras model
def create_model(input_dim):
    # create model
    model = Sequential()
    model.add(Dense(8, input_dim=input_dim, kernel_initializer='normal', activation='relu'))
    model.add(Dense(4, kernel_initializer='normal', activation='relu'))
    model.add(Dense(2, activation='sigmoid')) # output for 2 classes

    # compile model
    adam = Adam(learning_rate=0.001)
    model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=['accuracy'])
    return model

model = create_model(X_train.shape[1])
print(model.summary())

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 8)              │           712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │            36 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │            10 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 758 (2.96 KB)

 Trainable params: 758 (2.96 KB)

 Non-trainable params: 0 (0.00 B)

None


### 5. Training the Network

Now it's time for the fun! Training a Keras model is as simple as calling model.fit().

In [8]:
# 6. Train the Model
model.fit(X_train, Y_train, epochs=50, batch_size=10, verbose = 1)

Epoch 1/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.4592 - loss: nan
Epoch 2/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4807 - loss: nan
Epoch 3/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4807 - loss: nan
Epoch 4/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4807 - loss: nan
Epoch 5/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.4807 - loss: nan
Epoch 6/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4807 - loss: nan
Epoch 7/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.4807 - loss: nan
Epoch 8/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.4807 - loss: nan
Epoch 9/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4807 - loss: nan
Epoch 10/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.4807 - loss: nan
Epoch 11/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.4807 - loss: nan
Epoch 12/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4807 - loss: nan
Epoch 13/50
24/24 

### 6. Testing and Performance Metrics

Now that our model has been trained, we need to test its performance on the testing dataset. The model has never seen this information before; as a result, the testing dataset allows us to determine whether or not the model will be able to generalize to information that wasn't used during its training phase. We will use some of the metrics provided by scikit-learn for this purpose!

In [9]:
# 7. Evaluate the Model
# generate classification report using predictions for categorical model
# Use np.argmax for class prediction instead of deprecated predict_classes
pred_prob = model.predict(X_test)
pred_classes = np.argmax(pred_prob, axis=1)
true_classes = np.argmax(Y_test.values, axis=1)

print("Results for Categorical Model")
print("Accuracy:", accuracy_score(true_classes, pred_classes))
print(classification_report(true_classes, pred_classes))

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 177ms/step


NameError: name 'np' is not defined